In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash-lite')

In [ ]:
prompts = [
    {"role": "system", "content": "You are a advanced PHP developer, who knows the ins and out of that language"},
    {"role": "user", "content": "Explain any function or concept in 100 words"}
]

res = llm.invoke(prompts)

print(res.content)

# The problem here is that the prompt is static, if we are to change the variable PHP or user prompt, we would have to create this dict again

Let's talk about **Traits** in PHP. Traits are a mechanism for code reuse in single-inheritance languages like PHP. They allow you to group methods that can be included into classes. Unlike interfaces, traits can contain actual method implementations. A class can `use` multiple traits, inheriting their methods and properties. This provides a way to compose functionality horizontally across different class hierarchies without the limitations of traditional inheritance. Traits are declared using the `trait` keyword and used within classes with the `use` keyword, offering a powerful alternative for code organization and avoiding code duplication.


## Prompt Templates - Dynamic Prompts

In [24]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model = 'gpt-4o')
out = StrOutputParser()

In [ ]:
prompts = ChatPromptTemplate.from_messages([
    {"role": "system", "content": "You are an expert translator and translate input in {language}"},
    {"role": "user", "content": "{query}"}
])

print(prompts) # outputs a template of system message (prompts) and required input variables

input_variables=['language', 'query'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='You are an expert translator and translate input in {language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})]


In [ ]:
final_prompt = prompts.format_messages(language = "Hinglish", query = "I like LangChain")

print(final_prompt) # return an output that can be used to invoke the llm

[SystemMessage(content='You are an expert translator and translate input in Hinglish', additional_kwargs={}, response_metadata={}), HumanMessage(content='I like LangChain', additional_kwargs={}, response_metadata={})]


In [7]:
res = llm.invoke(final_prompt)

print(res.content)

Main LangChain pasand karta hoon.


In [13]:
# There another way to do this, the modern way using invoke directly (without format_messages)

final_prompt = prompts.invoke({
    "language": "Spanish",
    "query": "I want to learn LangChain and build something"
})

print(final_prompt) #this formats a similar system message that the format_message method does

messages=[SystemMessage(content='You are an expert translator and translate input in Spanish', additional_kwargs={}, response_metadata={}), HumanMessage(content='I want to learn LangChain and build something', additional_kwargs={}, response_metadata={})]


In [ ]:
res = llm.invoke(final_prompt)

print(res.content) # Passing the formatted message (final prompt) into the llm.invoke

Quiero aprender LangChain y construir algo.


## Dyanmic Prompts - Using Chains & Runnables

In [ ]:
prompts = ChatPromptTemplate.from_messages([
    {"role": "system", "content": "You are an expert phillospher who studys {phillosophy} and answers only related to that"},
    {"role": "user", "content": "{query}"}
])

chains = prompts | llm  | out # prompts and llm are defined, output after invoking prompts < goes into llm and it gets invoked, what we were doing above in one line

print(chains)

first=ChatPromptTemplate(input_variables=['phillosophy', 'query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['phillosophy'], input_types={}, partial_variables={}, template='You are an expert phillospher who studys {phillosophy} and answers only related to that'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})]) middle=[ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_output

In [ ]:
res = chains.invoke({
    "phillosophy": "Socratic Phillosophy",
    "query": "Explain the most complicated concept in simple 150-200 words"
})

print(res) # By using out as a runnable, we don't need to do res.content

One of the profound concepts in Socratic philosophy is the idea of Socratic irony. It's a method Socrates used in dialogue, which can initially seem complicated but is quite fascinating when unpacked.

Socratic irony involves Socrates pretending to be ignorant or claiming to know less than he actually does. He uses this humble stance to draw out knowledge and opinions from others. By doing so, he encourages his conversation partners to critically examine their own beliefs, often revealing contradictions or weaknesses in their thinking.

The brilliance of this technique lies in its ability to foster genuine understanding. Instead of directly telling people they are wrong, Socrates guides them to discover inconsistencies for themselves, prompting deeper reflection and learning.

Through irony, Socrates not only uncovers truth but also teaches humility—emphasizing the importance of recognizing one's own ignorance. This process of questioning and reflection helps individuals move closer to